# Классификация: SI выше медианы

Задача: предсказать, превышает ли SI медианное значение выборки (3.9).

Сравним модели на полном наборе из 210 признаков и сокращённом наборе из 158 признаков.

## 1. Импорты

In [19]:
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

## 2. Загрузка и подготовка

In [2]:
raw_data = pd.read_excel("../data/dataset.xlsx")
TARGET_COLUMNS = ["IC50, mM", "CC50, mM", "SI"]

data = raw_data.iloc[:, 1:].copy()
data = data.drop_duplicates()

feature_columns = [col for col in data.columns if col not in TARGET_COLUMNS]
X_full = data[feature_columns]

constant_mask = X_full.nunique(dropna=False) <= 1

dominant_share = X_full.apply(
    lambda column: column.value_counts(
        dropna=False,
        normalize=True,
    ).iloc[0]
)

near_constant_mask = dominant_share >= 0.95

X_reduced = X_full.loc[
    :,
    ~(constant_mask | near_constant_mask),
]

threshold = data["SI"].median()
y = (data["SI"] > threshold).astype(int)

print(f"Полный набор: {X_full.shape[1]} признаков")
print(f"Сокращённый: {X_reduced.shape[1]} признаков")
print(f"Класс 0: {(y == 0).sum()}, класс 1: {(y == 1).sum()} ({y.mean():.1%} положительных)")

Полный набор: 210 признаков
Сокращённый: 158 признаков
Класс 0: 485, класс 1: 484 (49.9% положительных)


## 3. Разбиение

Используем стратифицированное групповое разбиение. Объекты с одинаковыми значениями молекулярных дескрипторов относятся к одной группе и не могут одновременно попасть в обучение и тест.

Для подбора гиперпараметров также используем групповую стратифицированную кросс-валидацию.

In [3]:
groups = pd.util.hash_pandas_object(X_full, index=False)

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X_full, y, groups=groups)
)

Xf_train = X_full.iloc[train_idx]
Xf_test = X_full.iloc[test_idx]

Xr_train = X_reduced.iloc[train_idx]
Xr_test = X_reduced.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

print(f"Обучение: {len(Xf_train)}, тест: {len(Xf_test)}")

Обучение: 776, тест: 193


## 4. Метрики

Три метрики, каждая про своё:

- Accuracy — доля правильных ответов. При равных классах случайное угадывание даст 0.5
- F1 — среднее гармоническое точности и полноты положительного класса. Метрика учитывает как пропущенные положительные объекты, так и ложные положительные предсказания
- ROC-AUC — площадь под ROC-кривой. Показывает, насколько хорошо модель разделяет классы независимо от порога. 0.5 — случайно, 1.0 — идеально

Подбираем гиперпараметры по F1 — она требовательнее accuracy и лучше различает модели при кросс-валидации

Для сравнения используем baseline, который относит все объекты к положительному классу.

## 5. Логистическая регрессия

Линейный классификатор. Признаки масштабируем (StandardScaler) — без этого регуляризация давила бы на признаки неравномерно. Пропуски заполняем медианой.

Параметр C управляет силой регуляризации: меньше C — сильнее штраф и проще модель. Подбираем 5 значений.

In [4]:
lr_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(random_state=42, max_iter=5000))
lr_params = {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}

lr_search = GridSearchCV(lr_pipe, lr_params, cv=cv, scoring="f1", n_jobs=-1)
lr_search.fit(Xf_train, y_train, groups=groups_train)

lr_cv = pd.DataFrame({"C": lr_params["logisticregression__C"], "CV F1": lr_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(lr_cv)
print(f"Лучший C: {lr_search.best_params_['logisticregression__C']}, CV F1: {lr_search.best_score_:.4f}")

Полный набор


,C,CV F1
0,0.01,0.661005
1,0.10,0.658616
2,1.00,0.657405
3,10.00,0.629248
4,100.00,0.599960


Лучший C: 0.01, CV F1: 0.6610


In [5]:
lr_pred = lr_search.predict(Xf_test)
lr_proba = lr_search.predict_proba(Xf_test)[:, 1]
lr_full = {"name": "LR (полный)", "acc": accuracy_score(y_test, lr_pred), "f1": f1_score(y_test, lr_pred), "roc": roc_auc_score(y_test, lr_proba)}
print(f"Test Accuracy: {lr_full['acc']:.3f}, Test F1: {lr_full['f1']:.3f}, ROC-AUC: {lr_full['roc']:.3f}")

Test Accuracy: 0.622, Test F1: 0.614, ROC-AUC: 0.686


In [6]:
lr_search2 = GridSearchCV(make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(random_state=42, max_iter=5000)), lr_params, cv=cv, scoring="f1", n_jobs=-1)
lr_search2.fit(Xr_train, y_train, groups=groups_train)
lr_cv2 = pd.DataFrame({"C": lr_params["logisticregression__C"], "CV F1": lr_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(lr_cv2)
print(f"Лучший C: {lr_search2.best_params_['logisticregression__C']}, CV F1: {lr_search2.best_score_:.4f}")

lr_pred2 = lr_search2.predict(Xr_test)
lr_proba2 = lr_search2.predict_proba(Xr_test)[:, 1]
lr_reduced = {"name": "LR (сокращённый)", "acc": accuracy_score(y_test, lr_pred2), "f1": f1_score(y_test, lr_pred2), "roc": roc_auc_score(y_test, lr_proba2)}
print(f"Test Accuracy: {lr_reduced['acc']:.3f}, Test F1: {lr_reduced['f1']:.3f}, ROC-AUC: {lr_reduced['roc']:.3f}")

Сокращённый набор


,C,CV F1
0,0.01,0.672329
1,0.10,0.666224
2,1.00,0.655013
3,10.00,0.629881
4,100.00,0.632237


Лучший C: 0.01, CV F1: 0.6723
Test Accuracy: 0.580, Test F1: 0.521, ROC-AUC: 0.643


На полном наборе логистическая регрессия получила F1 = 0.614, на сокращённом — 0.521. Удаление признаков заметно ухудшило все три метрики.

## 6. K ближайших соседей

Голосование K ближайших соседей. StandardScaler критичен — иначе признаки с большим размахом задавят остальные при подсчёте расстояния. Подбираем K и тип взвешивания (uniform или distance).

In [7]:
knn_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier())
knn_params = {"kneighborsclassifier__n_neighbors": [3, 5, 7, 10, 15, 20], "kneighborsclassifier__weights": ["uniform", "distance"]}

knn_search = GridSearchCV(knn_pipe, knn_params, cv=cv, scoring="f1", n_jobs=-1)
knn_search.fit(Xf_train, y_train, groups=groups_train)

knn_cv = pd.DataFrame({"n_neighbors": knn_search.cv_results_["param_kneighborsclassifier__n_neighbors"], "weights": knn_search.cv_results_["param_kneighborsclassifier__weights"], "CV F1": knn_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(knn_cv.sort_values("CV F1", ascending=False))
print(f"Лучшие: {knn_search.best_params_}, CV F1: {knn_search.best_score_:.4f}")

Полный набор


,n_neighbors,weights,CV F1
9,15,distance,0.676882
8,15,uniform,0.674002
2,5,uniform,0.671663
11,20,distance,0.662263
3,5,distance,0.661250
4,7,uniform,0.652990
10,20,uniform,0.648510
0,3,uniform,0.647681
1,3,distance,0.646197
7,10,distance,0.645274


Лучшие: {'kneighborsclassifier__n_neighbors': 15, 'kneighborsclassifier__weights': 'distance'}, CV F1: 0.6769


In [8]:
knn_pred = knn_search.predict(Xf_test)
knn_proba = knn_search.predict_proba(Xf_test)[:, 1]
knn_full = {"name": "KNN (полный)", "acc": accuracy_score(y_test, knn_pred), "f1": f1_score(y_test, knn_pred), "roc": roc_auc_score(y_test, knn_proba)}
print(f"Test Accuracy: {knn_full['acc']:.3f}, Test F1: {knn_full['f1']:.3f}, ROC-AUC: {knn_full['roc']:.3f}")

Test Accuracy: 0.627, Test F1: 0.617, ROC-AUC: 0.696


In [9]:
knn_search2 = GridSearchCV(make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier()), knn_params, cv=cv, scoring="f1", n_jobs=-1)
knn_search2.fit(Xr_train, y_train, groups=groups_train)
knn_cv2 = pd.DataFrame({"n_neighbors": knn_search2.cv_results_["param_kneighborsclassifier__n_neighbors"], "weights": knn_search2.cv_results_["param_kneighborsclassifier__weights"], "CV F1": knn_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(knn_cv2.sort_values("CV F1", ascending=False))
print(f"Лучшие: {knn_search2.best_params_}, CV F1: {knn_search2.best_score_:.4f}")

knn_pred2 = knn_search2.predict(Xr_test)
knn_proba2 = knn_search2.predict_proba(Xr_test)[:, 1]
knn_reduced = {"name": "KNN (сокращённый)", "acc": accuracy_score(y_test, knn_pred2), "f1": f1_score(y_test, knn_pred2), "roc": roc_auc_score(y_test, knn_proba2)}
print(f"Test Accuracy: {knn_reduced['acc']:.3f}, Test F1: {knn_reduced['f1']:.3f}, ROC-AUC: {knn_reduced['roc']:.3f}")

Сокращённый набор


,n_neighbors,weights,CV F1
2,5,uniform,0.693912
11,20,distance,0.681837
7,10,distance,0.676220
3,5,distance,0.673319
9,15,distance,0.671997
10,20,uniform,0.666307
8,15,uniform,0.666016
4,7,uniform,0.662989
0,3,uniform,0.661641
6,10,uniform,0.657908


Лучшие: {'kneighborsclassifier__n_neighbors': 5, 'kneighborsclassifier__weights': 'uniform'}, CV F1: 0.6939
Test Accuracy: 0.689, Test F1: 0.659, ROC-AUC: 0.684


KNN получил F1 = 0.617 на полном наборе и 0.659 на сокращённом. Сокращение признаков улучшило Accuracy и F1, хотя ROC-AUC немного снизился.

## 7. Случайный лес

In [10]:
rf_pipe = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42))
rf_params = {"randomforestclassifier__n_estimators": [100, 200, 300, 500], "randomforestclassifier__max_depth": [10, 20, 30, None], "randomforestclassifier__min_samples_leaf": [1, 3, 5], "randomforestclassifier__max_features": ["sqrt", "log2", None]}

rf_search = GridSearchCV(rf_pipe, rf_params, cv=cv, scoring="f1", n_jobs=-1)
rf_search.fit(Xf_train, y_train, groups=groups_train)

rf_cv = pd.DataFrame({"n_estimators": rf_search.cv_results_["param_randomforestclassifier__n_estimators"], "max_depth": rf_search.cv_results_["param_randomforestclassifier__max_depth"].astype(str), "min_samples_leaf": rf_search.cv_results_["param_randomforestclassifier__min_samples_leaf"], "max_features": rf_search.cv_results_["param_randomforestclassifier__max_features"].astype(str), "CV F1": rf_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(rf_cv.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {rf_search.best_params_}, CV F1: {rf_search.best_score_:.4f}")

Полный набор


,n_estimators,max_depth,min_samples_leaf,max_features,CV F1
33,200,10,5,None,0.673033
105,200,30,5,None,0.672869
69,200,20,5,None,0.672869
141,200,None,5,None,0.672869
136,100,None,3,None,0.671635
64,100,20,3,None,0.671635
100,100,30,3,None,0.671635
71,500,20,5,None,0.671599
107,500,30,5,None,0.671599
143,500,None,5,None,0.671599


Лучшие: {'randomforestclassifier__max_depth': 10, 'randomforestclassifier__max_features': None, 'randomforestclassifier__min_samples_leaf': 5, 'randomforestclassifier__n_estimators': 200}, CV F1: 0.6730


In [11]:
rf_pred = rf_search.predict(Xf_test)
rf_proba = rf_search.predict_proba(Xf_test)[:, 1]
rf_full = {"name": "RF (полный)", "acc": accuracy_score(y_test, rf_pred), "f1": f1_score(y_test, rf_pred), "roc": roc_auc_score(y_test, rf_proba)}
print(f"Test Accuracy: {rf_full['acc']:.3f}, Test F1: {rf_full['f1']:.3f}, ROC-AUC: {rf_full['roc']:.3f}")

Test Accuracy: 0.617, Test F1: 0.543, ROC-AUC: 0.710


In [12]:
rf_search2 = GridSearchCV(make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42)), rf_params, cv=cv, scoring="f1", n_jobs=-1)
rf_search2.fit(Xr_train, y_train, groups=groups_train)
rf_cv2 = pd.DataFrame({"n_estimators": rf_search2.cv_results_["param_randomforestclassifier__n_estimators"], "max_depth": rf_search2.cv_results_["param_randomforestclassifier__max_depth"].astype(str), "min_samples_leaf": rf_search2.cv_results_["param_randomforestclassifier__min_samples_leaf"], "max_features": rf_search2.cv_results_["param_randomforestclassifier__max_features"].astype(str), "CV F1": rf_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(rf_cv2.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {rf_search2.best_params_}, CV F1: {rf_search2.best_score_:.4f}")

rf_pred2 = rf_search2.predict(Xr_test)
rf_proba2 = rf_search2.predict_proba(Xr_test)[:, 1]
rf_reduced = {"name": "RF (сокращённый)", "acc": accuracy_score(y_test, rf_pred2), "f1": f1_score(y_test, rf_pred2), "roc": roc_auc_score(y_test, rf_proba2)}
print(f"Test Accuracy: {rf_reduced['acc']:.3f}, Test F1: {rf_reduced['f1']:.3f}, ROC-AUC: {rf_reduced['roc']:.3f}")

Сокращённый набор


,n_estimators,max_depth,min_samples_leaf,max_features,CV F1
104,100,30,5,None,0.680159
140,100,None,5,None,0.680159
68,100,20,5,None,0.680159
32,100,10,5,None,0.678575
33,200,10,5,None,0.677139
69,200,20,5,None,0.676349
141,200,None,5,None,0.676349
105,200,30,5,None,0.676349
4,100,10,3,sqrt,0.675057
143,500,None,5,None,0.673959


Лучшие: {'randomforestclassifier__max_depth': 20, 'randomforestclassifier__max_features': None, 'randomforestclassifier__min_samples_leaf': 5, 'randomforestclassifier__n_estimators': 100}, CV F1: 0.6802
Test Accuracy: 0.601, Test F1: 0.522, ROC-AUC: 0.703


Случайный лес получил F1 = 0.543 на полном наборе и 0.522 на сокращённом. Сокращение признаков немного ухудшило качество модели.

## 8. Градиентный бустинг

In [13]:
gb_pipe = make_pipeline(SimpleImputer(strategy="median"), GradientBoostingClassifier(random_state=42))
gb_params = {"gradientboostingclassifier__n_estimators": [100, 200, 300], "gradientboostingclassifier__max_depth": [3, 5, 7], "gradientboostingclassifier__learning_rate": [0.01, 0.05, 0.1]}

gb_search = RandomizedSearchCV(gb_pipe, gb_params, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42)
gb_search.fit(Xf_train, y_train, groups=groups_train)

gb_cv = pd.DataFrame({"n_estimators": gb_search.cv_results_["param_gradientboostingclassifier__n_estimators"], "max_depth": gb_search.cv_results_["param_gradientboostingclassifier__max_depth"], "learning_rate": gb_search.cv_results_["param_gradientboostingclassifier__learning_rate"], "CV F1": gb_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(gb_cv.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {gb_search.best_params_}, CV F1: {gb_search.best_score_:.4f}")

Полный набор


,n_estimators,max_depth,learning_rate,CV F1
4,100,3,0.01,0.657149
2,100,3,0.05,0.640234
5,300,3,0.05,0.639983
13,300,3,0.01,0.638866
10,200,3,0.01,0.637499
12,300,5,0.01,0.635127
0,300,7,0.01,0.634128
6,200,7,0.05,0.620110
11,200,5,0.01,0.619447
19,100,3,0.10,0.616968


Лучшие: {'gradientboostingclassifier__n_estimators': 100, 'gradientboostingclassifier__max_depth': 3, 'gradientboostingclassifier__learning_rate': 0.01}, CV F1: 0.6571


In [14]:
gb_pred = gb_search.predict(Xf_test)
gb_proba = gb_search.predict_proba(Xf_test)[:, 1]
gb_full = {"name": "GBT (полный)", "acc": accuracy_score(y_test, gb_pred), "f1": f1_score(y_test, gb_pred), "roc": roc_auc_score(y_test, gb_proba)}
print(f"Test Accuracy: {gb_full['acc']:.3f}, Test F1: {gb_full['f1']:.3f}, ROC-AUC: {gb_full['roc']:.3f}")

Test Accuracy: 0.611, Test F1: 0.497, ROC-AUC: 0.679


In [15]:
gb_search2 = RandomizedSearchCV(make_pipeline(SimpleImputer(strategy="median"), GradientBoostingClassifier(random_state=42)), gb_params, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42)
gb_search2.fit(Xr_train, y_train, groups=groups_train)
gb_cv2 = pd.DataFrame({"n_estimators": gb_search2.cv_results_["param_gradientboostingclassifier__n_estimators"], "max_depth": gb_search2.cv_results_["param_gradientboostingclassifier__max_depth"], "learning_rate": gb_search2.cv_results_["param_gradientboostingclassifier__learning_rate"], "CV F1": gb_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(gb_cv2.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {gb_search2.best_params_}, CV F1: {gb_search2.best_score_:.4f}")

gb_pred2 = gb_search2.predict(Xr_test)
gb_proba2 = gb_search2.predict_proba(Xr_test)[:, 1]
gb_reduced = {"name": "GBT (сокращённый)", "acc": accuracy_score(y_test, gb_pred2), "f1": f1_score(y_test, gb_pred2), "roc": roc_auc_score(y_test, gb_proba2)}
print(f"Test Accuracy: {gb_reduced['acc']:.3f}, Test F1: {gb_reduced['f1']:.3f}, ROC-AUC: {gb_reduced['roc']:.3f}")

Сокращённый набор


,n_estimators,max_depth,learning_rate,CV F1
4,100,3,0.01,0.648161
5,300,3,0.05,0.643610
2,100,3,0.05,0.641766
13,300,3,0.01,0.639699
14,100,7,0.05,0.637737
11,200,5,0.01,0.634276
10,200,3,0.01,0.633576
0,300,7,0.01,0.632851
7,300,7,0.05,0.631829
18,300,5,0.10,0.628809


Лучшие: {'gradientboostingclassifier__n_estimators': 100, 'gradientboostingclassifier__max_depth': 3, 'gradientboostingclassifier__learning_rate': 0.01}, CV F1: 0.6482
Test Accuracy: 0.611, Test F1: 0.497, ROC-AUC: 0.679


Градиентный бустинг получил одинаковый F1 = 0.497 на полном и сокращённом наборах. Удаление признаков не повлияло на итоговые метрики.

## 9. XGBoost

Реализация градиентного бустинга с дополнительной регуляризацией.

In [16]:
xgb_pipe = make_pipeline(SimpleImputer(strategy="median"), XGBClassifier(random_state=42, eval_metric="logloss"))
xgb_params = {"xgbclassifier__n_estimators": [100, 200, 300], "xgbclassifier__max_depth": [3, 5, 7], "xgbclassifier__learning_rate": [0.01, 0.05, 0.1]}

xgb_search = RandomizedSearchCV(xgb_pipe, xgb_params, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42)
xgb_search.fit(Xf_train, y_train, groups=groups_train)

xgb_cv = pd.DataFrame({"n_estimators": xgb_search.cv_results_["param_xgbclassifier__n_estimators"], "max_depth": xgb_search.cv_results_["param_xgbclassifier__max_depth"], "learning_rate": xgb_search.cv_results_["param_xgbclassifier__learning_rate"], "CV F1": xgb_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(xgb_cv.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {xgb_search.best_params_}, CV F1: {xgb_search.best_score_:.4f}")

xgb_pred = xgb_search.predict(Xf_test)
xgb_proba = xgb_search.predict_proba(Xf_test)[:, 1]
xgb_full = {"name": "XGB (полный)", "acc": accuracy_score(y_test, xgb_pred), "f1": f1_score(y_test, xgb_pred), "roc": roc_auc_score(y_test, xgb_proba)}
print(f"Test Accuracy: {xgb_full['acc']:.3f}, Test F1: {xgb_full['f1']:.3f}, ROC-AUC: {xgb_full['roc']:.3f}")

xgb_search2 = RandomizedSearchCV(make_pipeline(SimpleImputer(strategy="median"), XGBClassifier(random_state=42, eval_metric="logloss")), xgb_params, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42)
xgb_search2.fit(Xr_train, y_train, groups=groups_train)
xgb_cv2 = pd.DataFrame({"n_estimators": xgb_search2.cv_results_["param_xgbclassifier__n_estimators"], "max_depth": xgb_search2.cv_results_["param_xgbclassifier__max_depth"], "learning_rate": xgb_search2.cv_results_["param_xgbclassifier__learning_rate"], "CV F1": xgb_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(xgb_cv2.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {xgb_search2.best_params_}, CV F1: {xgb_search2.best_score_:.4f}")

xgb_pred2 = xgb_search2.predict(Xr_test)
xgb_proba2 = xgb_search2.predict_proba(Xr_test)[:, 1]
xgb_reduced = {"name": "XGB (сокращённый)", "acc": accuracy_score(y_test, xgb_pred2), "f1": f1_score(y_test, xgb_pred2), "roc": roc_auc_score(y_test, xgb_proba2)}
print(f"Test Accuracy: {xgb_reduced['acc']:.3f}, Test F1: {xgb_reduced['f1']:.3f}, ROC-AUC: {xgb_reduced['roc']:.3f}")

Полный набор


,n_estimators,max_depth,learning_rate,CV F1
4,100,3,0.01,0.659349
0,300,7,0.01,0.655720
2,100,3,0.05,0.654783
13,300,3,0.01,0.651499
10,200,3,0.01,0.642351
19,100,3,0.10,0.639352
11,200,5,0.01,0.637696
16,100,5,0.01,0.635187
12,300,5,0.01,0.632868
1,200,5,0.05,0.630666


Лучшие: {'xgbclassifier__n_estimators': 100, 'xgbclassifier__max_depth': 3, 'xgbclassifier__learning_rate': 0.01}, CV F1: 0.6593
Test Accuracy: 0.617, Test F1: 0.493, ROC-AUC: 0.681
Сокращённый набор


,n_estimators,max_depth,learning_rate,CV F1
4,100,3,0.01,0.659349
2,100,3,0.05,0.654826
13,300,3,0.01,0.651499
0,300,7,0.01,0.646176
10,200,3,0.01,0.642351
19,100,3,0.10,0.641164
11,200,5,0.01,0.637696
5,300,3,0.05,0.636126
16,100,5,0.01,0.635187
12,300,5,0.01,0.631956


Лучшие: {'xgbclassifier__n_estimators': 100, 'xgbclassifier__max_depth': 3, 'xgbclassifier__learning_rate': 0.01}, CV F1: 0.6593
Test Accuracy: 0.617, Test F1: 0.493, ROC-AUC: 0.681


XGBoost получил одинаковые результаты на обоих наборах: F1 = 0.493 и ROC-AUC = 0.681. По F1 модель немного уступила GradientBoostingClassifier.

## 10. Сравнение

Сравним все модели с наивным baseline.

In [20]:
baseline_pred = np.ones(len(y_test), dtype=int)
baseline_proba = np.ones(len(y_test))

baseline = {
    "name": "Baseline (все 1)",
    "acc": accuracy_score(y_test, baseline_pred),
    "f1": f1_score(y_test, baseline_pred),
    "roc": roc_auc_score(y_test, baseline_proba),
}

print(
    f"Baseline | Accuracy: {baseline['acc']:.3f}, "
    f"F1: {baseline['f1']:.3f}, "
    f"ROC-AUC: {baseline['roc']:.3f}"
)

Baseline | Accuracy: 0.497, F1: 0.664, ROC-AUC: 0.500


In [22]:
res_full = pd.DataFrame(
    [baseline, lr_full, knn_full, rf_full, gb_full, xgb_full]
).sort_values("f1", ascending=False).reset_index(drop=True)

res_full.columns = ["Модель", "Accuracy", "F1", "ROC-AUC"]

print("210 признаков")
display(res_full.round(3))

res_reduced = pd.DataFrame(
    [
        baseline,
        lr_reduced,
        knn_reduced,
        rf_reduced,
        gb_reduced,
        xgb_reduced,
    ]
).sort_values("f1", ascending=False).reset_index(drop=True)

res_reduced.columns = ["Модель", "Accuracy", "F1", "ROC-AUC"]

print("158 признаков")
display(res_reduced.round(3))

=== 210 признаков ===


,Модель,Accuracy,F1,ROC-AUC
0,Baseline (все 1),0.497,0.664,0.500
1,KNN (полный),0.627,0.617,0.696
2,LR (полный),0.622,0.614,0.686
3,RF (полный),0.617,0.543,0.710
4,GBT (полный),0.611,0.497,0.679
5,XGB (полный),0.617,0.493,0.681


=== 158 признаков ===


,Модель,Accuracy,F1,ROC-AUC
0,Baseline (все 1),0.497,0.664,0.500
1,KNN (сокращённый),0.689,0.659,0.684
2,RF (сокращённый),0.601,0.522,0.703
3,LR (сокращённый),0.580,0.521,0.643
4,GBT (сокращённый),0.611,0.497,0.679
5,XGB (сокращённый),0.617,0.493,0.681


## 11. Выводы

Лучший результат среди обученных моделей показал KNN на сокращённом наборе признаков: F1 = 0.659 и Accuracy = 0.689. На полном наборе KNN получил F1 = 0.617.

Baseline, который относит все объекты к положительному классу, получил F1 = 0.664. Таким образом, ни одна обученная модель не превзошла baseline по F1. При этом KNN на сокращённом наборе получил значительно более высокую Accuracy, а случайный лес на полном наборе показал лучший ROC-AUC = 0.710.

Удаление константных и почти константных признаков помогло только KNN. Для логистической регрессии и случайного леса качество снизилось, а результаты градиентного бустинга и XGBoost практически не изменились.

Для итогового решения можно выбрать KNN на сокращённом наборе как наиболее сбалансированную модель. Однако результаты показывают, что классификация SI выше медианы остаётся сложной задачей и требует дополнительных признаков или других способов представления молекул.

Групповое разбиение исключает попадание объектов с одинаковыми дескрипторами одновременно в обучение и тест.